# AdaptiveAI backend on Kaggle (strong model + ngrok)
Runs your **real project backend** (`app.py` from `ddd.zip`) so the Streamlit app works fully:
`/health /ingest /topics /chat /quiz /flashcards /reset`.

**Before running**
1. Notebook settings → **Accelerator: GPU T4 x2**, **Internet: On**.
2. Upload `ddd.zip` via *Add Input → Upload → New Dataset* (or drop it in `/kaggle/working`).
3. *Add-ons → Secrets*: add `NGROK_TOKEN` (from dashboard.ngrok.com) and `API_KEY` (any long private password).
4. Run cells top to bottom. Copy the printed URL into the Streamlit sidebar, and use the same API key.

In [9]:
# 1) Pick the model. Bigger = better answers/quizzes but slower and longer first download.
#   "Qwen/Qwen2.5-7B-Instruct"   -> fast, good
#   "Qwen/Qwen2.5-14B-Instruct"  -> recommended (best quality/speed on T4 x2, 4-bit)
#   "Qwen/Qwen2.5-32B-Instruct"  -> strongest, ~19GB in 4-bit, slow on T4s
LLM_NAME = "Qwen/Qwen2.5-14B-Instruct"
QUANT = "4bit"

In [10]:
# 2) Find the project (works whether Kaggle kept the zip or unpacked it)
import glob, os, shutil, subprocess, sys

print("Contents of /kaggle/input:")
!find /kaggle/input -maxdepth 3 | head -40

shutil.rmtree("/kaggle/working/proj", ignore_errors=True)
os.makedirs("/kaggle/working/proj", exist_ok=True)

zips = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/kaggle/working/*.zip")
apps = glob.glob("/kaggle/input/**/app.py", recursive=True)

if apps:
    # Dataset was auto-unzipped: copy the project to a writable place
    src = os.path.dirname(apps[0])
    shutil.copytree(src, "/kaggle/working/proj/ddd")
elif zips:
    subprocess.run(["unzip", "-o", "-q", zips[0], "-d", "/kaggle/working/proj"], check=True)
else:
    raise FileNotFoundError(
        "No project found. Make sure the dataset is attached under Input "
        "(right sidebar). See the listing above."
    )

PROJECT_DIR = os.path.dirname(glob.glob("/kaggle/working/proj/**/app.py", recursive=True)[0])
print("Project dir:", PROJECT_DIR)

Contents of /kaggle/input:
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/janakassem91193
/kaggle/input/datasets/janakassem91193/studybuddy
Project dir: /kaggle/working/proj/ddd


In [5]:
# 3) Install dependencies (keeps Kaggle's own torch/numpy)
reqs = [l for l in open(f"{PROJECT_DIR}/requirements.txt")
        if l.strip() and not l.startswith("#") and not l.startswith("numpy")]
open("/tmp/reqs.txt", "w").writelines(reqs)
!pip install -q -r /tmp/reqs.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 71.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.9/443.9 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [11]:
# 4) Secrets + environment (config.py reads these)
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    NGROK_TOKEN = s.get_secret("NGROK_TOKEN")
    API_KEY = s.get_secret("API_KEY")
except Exception:
    NGROK_TOKEN = "PASTE_NGROK_TOKEN"      # fallback if you don't use Secrets
    API_KEY = "CHANGE_ME_LONG_PASSWORD"    # never share the notebook with real values here

os.environ["API_KEY"] = API_KEY
os.environ["LLM_NAME"] = LLM_NAME
os.environ["QUANT"] = QUANT
os.makedirs("/kaggle/temp/hf", exist_ok=True)
os.environ["HF_HOME"] = "/kaggle/temp/hf"   # big disk for model downloads
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

In [ ]:
# 5) Load the model + backend (first run downloads the model: several minutes)
from app import app
import torch
print("GPUs:", torch.cuda.device_count())

[AdaptiveAI] Loading language model: Qwen/Qwen2.5-14B-Instruct (4bit)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00005-of-00008.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00001-of-00008.safetensors:   0%|          | 0.00/3.89G [00:00<?, ?B/s]

model-00003-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00008-of-00008.safetensors:   0%|          | 0.00/1.70G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [12]:
# 6) Start the server and the ngrok tunnel
import threading, time, socket, uvicorn
from pyngrok import ngrok, conf

PORT = 8000
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()                      # close tunnels from earlier runs
public_url = ngrok.connect(PORT).public_url

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info"),
    daemon=True,
).start()
time.sleep(3)

print("\nPUBLIC URL  :", public_url)
print("API KEY     : (the API_KEY secret you set)")
print("Docs        :", public_url + "/docs")

INFO:     Started server process [48]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



PUBLIC URL  : https://require-spectrum-probiotic.ngrok-free.dev
API KEY     : (the API_KEY secret you set)
Docs        : https://require-spectrum-probiotic.ngrok-free.dev/docs


In [9]:
# 7) Self-test: should show status ok and your model name
import requests
r = requests.get(public_url + "/health", headers={"ngrok-skip-browser-warning": "1"})
print(r.json())
r = requests.post(public_url + "/generate" if False else public_url + "/chat",
                  headers={"Authorization": f"Bearer {API_KEY}", "ngrok-skip-browser-warning": "1"},
                  json={"question": "test", "k": 1})
print(r.status_code, r.text[:200])   # 400 'No PDF material indexed yet' = auth works

INFO:     34.10.220.186:0 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok', 'model': 'Qwen/Qwen2.5-14B-Instruct', 'quantization': '4bit', 'model_memory_footprint_gb': 9.05, 'gpu_allocated_memory_gb_by_device': {'cuda:0': 5.43, 'cuda:1': 5.97}, 'chunks_indexed': 0, 'documents': []}
INFO:     34.10.220.186:0 - "POST /chat HTTP/1.1" 400 Bad Request
400 {"detail":"No PDF material indexed yet. Call /ingest first."}
INFO:     41.35.37.205:0 - "GET / HTTP/1.1" 200 OK
INFO:     41.35.37.205:0 - "GET / HTTP/1.1" 200 OK
INFO:     41.35.37.205:0 - "GET / HTTP/1.1" 200 OK
INFO:     41.35.37.205:0 - "GET /health HTTP/1.1" 200 OK
INFO:     41.35.37.205:0 - "POST /reset HTTP/1.1" 401 Unauthorized
INFO:     41.35.37.205:0 - "POST /reset HTTP/1.1" 401 Unauthorized
INFO:     41.35.37.205:0 - "POST /ingest HTTP/1.1" 401 Unauthorized
